# APTOS 2019 — Fundus Disease Detection Baseline

**Pipeline**: EfficientNet-B7 / Swin-Base → Ordinal Regression → Quadratic Weighted Kappa  
**Hardware**: 2× T4 GPU (Kaggle) với DDP + Mixed Precision  
**Dataset**: [APTOS 2019 Blindness Detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection)

---
| Cell | Chức năng |
|------|----------|
| 1 | Cài đặt thư viện & import modules |
| 2 | **Config** — chỉnh toàn bộ thông số tại đây |
| 3 | Dataset & DataLoader |
| 4 | Khởi tạo model |
| 5 | Training (DDP + AMP) |
| 6 | Evaluation (held-out 10%) |
| 7 | Generate submission.csv |
| 8 | Vẽ figures & zip outputs |

In [ ]:
%%time
# ── Cell 1: Cài đặt thư viện & import ────────────────────────────────────────
# Cài đặt các thư viện cần thiết (bỏ comment nếu chạy lần đầu trên Kaggle)

# !pip install -q timm wandb --upgrade

import os
import sys
import json
import torch
import pandas as pd
import numpy as np

# Thêm thư mục src vào Python path để import được các module
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.dataset   import get_dataloaders
from src.models    import build_model, predict_labels
from src.train     import run_training
from src.evaluate  import run_evaluation
from src.visualize import plot_all
from src.utils     import load_checkpoint, generate_submission, zip_outputs, load_history

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU count       : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
%%time
# ── Cell 2: CONFIG — Chỉnh toàn bộ thông số tại đây ──────────────────────────
# Đây là cell DUY NHẤT cần chỉnh sửa để thay đổi thông số thực nghiệm.

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────────────────
MODEL_TYPE  = "efficientnet_b7"   # "efficientnet_b7" | "swin_transformer"
IMAGE_SIZE  = 456                 # 456 cho efficientnet_b7 | 224 cho swin_transformer

# ─────────────────────────────────────────────────────────────────────────────
#  TRAINING
# ─────────────────────────────────────────────────────────────────────────────
BATCH_SIZE  = 16                  # per GPU (tổng effective = BATCH_SIZE × số GPU)
LR          = 1e-4                # Learning rate Adam
EPOCHS      = 15                  # Số epoch huấn luyện
USE_AMP     = True                # Mixed Precision FP16 (T4 hỗ trợ tốt)
NUM_WORKERS = 4                   # DataLoader workers per process

# ─────────────────────────────────────────────────────────────────────────────
#  DATA SPLIT  (tổng = 1.0)
# ─────────────────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.7                 # 70% train
VAL_RATIO   = 0.2                 # 20% validation (lưu best model)
TEST_RATIO  = 0.1                 # 10% held-out internal test (báo cáo metrics)
SEED        = 42

# ─────────────────────────────────────────────────────────────────────────────
#  PATHS  (Kaggle environment)
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR   = "/kaggle/input/competitions/aptos2019-blindness-detection"
OUTPUT_DIR = "/kaggle/working/outputs"

# ─────────────────────────────────────────────────────────────────────────────
#  WEIGHTS & BIASES
#  → Thêm WANDB_API_KEY vào Kaggle Secrets (Add-ons > Secrets)
# ─────────────────────────────────────────────────────────────────────────────
WANDB_PROJECT  = "fundus-baseline"
WANDB_RUN_NAME = f"{MODEL_TYPE}_img{IMAGE_SIZE}_bs{BATCH_SIZE}_lr{LR}_ep{EPOCHS}"

# Lấy key từ Kaggle Secrets (an toàn hơn hardcode)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    pass  # Nếu không có Kaggle Secrets thì bỏ qua — W&B sẽ bị vô hiệu hóa

# ─────────────────────────────────────────────────────────────────────────────
#  Tạo thư mục output
# ─────────────────────────────────────────────────────────────────────────────
os.makedirs(os.path.join(OUTPUT_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "figures"),     exist_ok=True)

# Đóng gói toàn bộ config vào dict để truyền vào các hàm src/
CFG = {
    "MODEL_TYPE":     MODEL_TYPE,
    "IMAGE_SIZE":     IMAGE_SIZE,
    "BATCH_SIZE":     BATCH_SIZE,
    "LR":             LR,
    "EPOCHS":         EPOCHS,
    "USE_AMP":        USE_AMP,
    "NUM_WORKERS":    NUM_WORKERS,
    "TRAIN_RATIO":    TRAIN_RATIO,
    "VAL_RATIO":      VAL_RATIO,
    "SEED":           SEED,
    "DATA_DIR":       DATA_DIR,
    "OUTPUT_DIR":     OUTPUT_DIR,
    "WANDB_PROJECT":  WANDB_PROJECT,
    "WANDB_RUN_NAME": WANDB_RUN_NAME,
}

print("Config:")
for k, v in CFG.items():
    print(f"  {k:<18} = {v}")

In [ ]:
%%time
# ── Cell 3: Dataset & DataLoader ─────────────────────────────────────────────
# Tạo 4 DataLoader từ dataset:
#   train_loader        : 70% train.csv — dùng DistributedSampler khi DDP
#   val_loader          : 20% train.csv — monitor val QWK, lưu best model
#   internal_test_loader: 10% train.csv — held-out, báo cáo metrics cuối
#   submit_loader       : 100% test.csv (không nhãn) — inference → submission.csv

# DataLoader ở đây chạy trên CPU/single-GPU để preview;
# khi training DDP, train_worker() sẽ tạo lại DataLoader với DistributedSampler.
(
    train_loader,
    val_loader,
    internal_test_loader,
    submit_loader,
    _,
) = get_dataloaders(
    data_dir    = DATA_DIR,
    image_size  = IMAGE_SIZE,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = SEED,
    rank        = 0,
    world_size  = 1,   # single process cho preview
)

# Lấy danh sách image_ids của tập submit để dùng ở Cell 7
test_csv   = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
SUBMIT_IDS = test_csv["id_code"].tolist()

print(f"Train batches        : {len(train_loader):>5}  (~{len(train_loader.dataset)} ảnh)")
print(f"Val batches          : {len(val_loader):>5}  (~{len(val_loader.dataset)} ảnh)")
print(f"Internal test batches: {len(internal_test_loader):>5}  (~{len(internal_test_loader.dataset)} ảnh)")
print(f"Submit batches       : {len(submit_loader):>5}  (~{len(submit_loader.dataset)} ảnh)")

# Kiểm tra shape batch đầu tiên
sample_imgs, sample_labels = next(iter(train_loader))
print(f"\nSample batch — images: {sample_imgs.shape}, labels: {sample_labels.shape}")
print(f"Label range: {sample_labels.min().item():.0f} – {sample_labels.max().item():.0f}")

In [ ]:
%%time
# ── Cell 4: Khởi tạo Model ────────────────────────────────────────────────────
# Tải pretrained ImageNet weights từ HuggingFace Hub qua timm.
# Classifier head được thay bằng 1 neuron cho ordinal regression.

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model  = build_model(MODEL_TYPE, pretrained=True).to(device)

# Đếm tham số
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model type       : {MODEL_TYPE}")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")
print(f"Device           : {device}")

# Kiểm tra forward pass với 1 batch nhỏ
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    out   = model(dummy)
    print(f"Forward pass OK  : input {tuple(dummy.shape)} → output {tuple(out.shape)}")

In [ ]:
%%time
# ── Cell 5: Training ──────────────────────────────────────────────────────────
# Khởi động DDP training trên 2× T4 GPU bằng mp.spawn.
# Mỗi epoch in: time | train loss/acc/QWK | val loss/acc/QWK
# Best model (theo val QWK) được lưu tự động vào OUTPUT_DIR/checkpoints/
# W&B log đầy đủ (rank 0 only) — theo dõi tại wandb.ai

run_training(CFG)

In [ ]:
%%time
# ── Cell 6: Evaluation trên Internal Test Set (held-out 10%) ─────────────────
# Load best checkpoint rồi đánh giá trên tập test nội bộ (có nhãn).
# In đầy đủ: QWK, Accuracy, Macro F1, Balanced Accuracy, Per-class Recall,
#            Confusion Matrix, Classification Report
# Lưu classification_report.txt vào OUTPUT_DIR

import torch.nn as nn

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Tạo lại model và load best checkpoint
eval_model = build_model(MODEL_TYPE, pretrained=False).to(device)
ckpt_path  = os.path.join(OUTPUT_DIR, "checkpoints", f"best_model_{MODEL_TYPE}.pth")
ckpt_meta  = load_checkpoint(eval_model, ckpt_path, device)

criterion = nn.SmoothL1Loss()

# Tạo lại internal_test_loader (single process, no DDP)
_, _, eval_test_loader, _, _ = get_dataloaders(
    data_dir    = DATA_DIR,
    image_size  = IMAGE_SIZE,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = SEED,
    rank        = 0,
    world_size  = 1,
)

# Chạy evaluation — in kết quả và lưu report
EVAL_METRICS = run_evaluation(
    model      = eval_model,
    loader     = eval_test_loader,
    criterion  = criterion,
    device     = device,
    output_dir = OUTPUT_DIR,
)

In [ ]:
%%time
# ── Cell 7: Generate Submission CSV ──────────────────────────────────────────
# Chạy inference trên test.csv (không nhãn) với best model.
# Lưu submission.csv đúng format của Kaggle (id_code, diagnosis).

# Tạo lại submit_loader (no DDP)
_, _, _, sub_loader, _ = get_dataloaders(
    data_dir    = DATA_DIR,
    image_size  = IMAGE_SIZE,
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = SEED,
    rank        = 0,
    world_size  = 1,
)

sub_path = generate_submission(
    model        = eval_model,
    submit_loader= sub_loader,
    image_ids    = SUBMIT_IDS,
    output_dir   = OUTPUT_DIR,
    device       = device,
)

# Preview submission
sub_df = pd.read_csv(sub_path)
print(f"\nSubmission preview ({len(sub_df)} rows):")
print(sub_df.head(10).to_string(index=False))
print(f"\nDistribution:\n{sub_df['diagnosis'].value_counts().sort_index().to_string()}")

In [ ]:
%%time
# ── Cell 8: Vẽ Figures & Zip Outputs ─────────────────────────────────────────
# Tạo 3 figure đánh giá:
#   1. Confusion Matrix (normalized)
#   2. Training Curves (Loss + QWK theo epoch)
#   3. Per-class Recall bar chart
# Sau đó zip toàn bộ outputs/ thành outputs.zip

import matplotlib
matplotlib.use('Agg')   # non-interactive backend cho Kaggle
import matplotlib.pyplot as plt

# Đọc history từ file JSON đã lưu trong quá trình training
HISTORY = load_history(OUTPUT_DIR)

# Vẽ và lưu tất cả figure
saved_figures = plot_all(
    metrics    = EVAL_METRICS,
    history    = HISTORY,
    output_dir = OUTPUT_DIR,
    model_type = MODEL_TYPE,
)

# Hiển thị figure trong notebook
import matplotlib.image as mpimg
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
titles = ["Confusion Matrix", "Training Curves", "Per-class Recall"]
for ax, path, title in zip(axes, saved_figures, titles):
    img = mpimg.imread(path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Zip toàn bộ outputs
zip_path = zip_outputs(OUTPUT_DIR)

print("\n" + "="*50)
print("PIPELINE HOÀN TẤT")
print("="*50)
print(f"  Best model  : {os.path.join(OUTPUT_DIR, 'checkpoints', f'best_model_{MODEL_TYPE}.pth')}")
print(f"  Submission  : {os.path.join(OUTPUT_DIR, 'submission.csv')}")
print(f"  Report      : {os.path.join(OUTPUT_DIR, 'classification_report.txt')}")
print(f"  Figures     : {os.path.join(OUTPUT_DIR, 'figures/')}")
print(f"  Archive     : {zip_path}")
print(f"\n  QWK (held-out test) : {EVAL_METRICS['qwk']:.4f}")
print(f"  Accuracy            : {EVAL_METRICS['accuracy']:.4f}")
print(f"  Macro F1            : {EVAL_METRICS['macro_f1']:.4f}")
print(f"  Balanced Accuracy   : {EVAL_METRICS['balanced_accuracy']:.4f}")